In [ ]:
#cell 1
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

np.random.seed(42)
tf.random.set_seed(42)

CONFIG = {
    "sequence_length": 32,
    "prediction_steps": 1,
    "batch_size": 32,
    "learning_rate": 0.001,

    # MODEL UNITS
    "lstm_units": 40,
    "gru_units": 40,
    "bilstm_units": 40,
    "bigru_units": 40
}

TARGET_COL = "CPU usage [%]"

#Creating groups
TOTAL_FILES = 519
GROUP_SIZE = 5

file_groups = []

for start in range(1, TOTAL_FILES+1, GROUP_SIZE):

    group = list(range(start, start + GROUP_SIZE))

    if group[-1] <= TOTAL_FILES:
        file_groups.append(group)


print("Total runs:", len(file_groups))
print("Example groups:", file_groups[:10])

In [ ]:
for CLIENT_FILE_NUMBERS in file_groups:

    print("\n=======================================")
    print("Running experiment for files:", CLIENT_FILE_NUMBERS)
    print("=======================================\n")


    #cell 2
    DATA_DIR = "/kaggle/input/datasets/darshanhegde17/materna-t13/Materna-Trace-1"  
    
    all_csv_files = sorted(
        glob.glob(os.path.join(DATA_DIR, "*.csv")),
        key=lambda x: int(os.path.basename(x).replace(".csv",""))
    )
    
    csv_files = [all_csv_files[i-1] for i in CLIENT_FILE_NUMBERS]


    #cell 3
    def load_client_data(path):
    
        df = pd.read_csv(path, sep=";", low_memory=False)
    
        df[TARGET_COL] = (
            df[TARGET_COL].astype(str)
            .str.replace(",", ".")
            .astype(float)
        )
    
        df = df.ffill().bfill()
    
        y_raw = df[[TARGET_COL]].values
    
        return y_raw
    
    
    def make_sequences(y, L, H):
    
        Xs, ys = [], []
    
        for i in range(len(y)-L-H):
            Xs.append(y[i:i+L])
            ys.append(y[i+L:i+L+H].flatten())
    
        return np.array(Xs), np.array(ys)


    #cell 4
    clients_data = []
    
    for path in csv_files:
    
        # LOAD RAW DATA (NO SCALING)
        y_raw = load_client_data(path)
    
        n_total = len(y_raw)
    
        # SPLIT FIRST (RAW DATA)
        train_end = int(0.6 * n_total)
        val_end   = int(0.8 * n_total)
    
        y_train_raw = y_raw[:train_end]
        y_val_raw   = y_raw[train_end:val_end]
        y_test_raw  = y_raw[val_end:]
    
        # SCALE (FIT ONLY ON TRAIN)
        scaler = MinMaxScaler()
    
        scaler.fit(y_train_raw)
    
        y_train_scaled = scaler.transform(y_train_raw)
        y_val_scaled   = scaler.transform(y_val_raw)
        y_test_scaled  = scaler.transform(y_test_raw)
    
        # CREATE SEQUENCES SEPARATELY
        X_train, y_train = make_sequences(
            y_train_scaled,
            CONFIG["sequence_length"],
            CONFIG["prediction_steps"]
        )
    
        X_val, y_val = make_sequences(
            y_val_scaled,
            CONFIG["sequence_length"],
            CONFIG["prediction_steps"]
        )
    
        X_test, y_test = make_sequences(
            y_test_scaled,
            CONFIG["sequence_length"],
            CONFIG["prediction_steps"]
        )
    
        clients_data.append({
            "X_train": X_train,
            "y_train": y_train,
            "X_val": X_val,
            "y_val": y_val,
            "X_test": X_test,
            "y_test": y_test,
            "scaler": scaler
        })
    
    # CONCATENATE
    X_train_all = np.concatenate([c["X_train"] for c in clients_data])
    y_train_all = np.concatenate([c["y_train"] for c in clients_data])
    
    X_val_all = np.concatenate([c["X_val"] for c in clients_data])
    y_val_all = np.concatenate([c["y_val"] for c in clients_data])


    #cell 5
    def build_LSTM():
    
        u = CONFIG["lstm_units"]
    
        model = Sequential([
            LSTM(u, activation="tanh", return_sequences=True,
                 input_shape=(CONFIG["sequence_length"],1)),
            LSTM(u, activation="tanh", return_sequences=True),
            LSTM(u, activation="tanh"),
            Dense(CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    def build_GRU():
    
        u = CONFIG["gru_units"]
    
        model = Sequential([
            GRU(u, activation="tanh", return_sequences=True,
                input_shape=(CONFIG["sequence_length"],1)),
            GRU(u, activation="tanh", return_sequences=True),
            GRU(u, activation="tanh"),
            Dense(CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    
    def build_BiLSTM():
    
        u = CONFIG["bilstm_units"]
    
        model = Sequential([
            Bidirectional(LSTM(u, activation="tanh", return_sequences=True),
                          input_shape=(CONFIG["sequence_length"],1)),
            Bidirectional(LSTM(u, activation="tanh", return_sequences=True)),
            Bidirectional(LSTM(u, activation="tanh")),
            Dense(CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    
    def build_BiGRU():
    
        u = CONFIG["bigru_units"]
    
        model = Sequential([
            Bidirectional(GRU(u, activation="tanh", return_sequences=True),
                          input_shape=(CONFIG["sequence_length"],1)),
            Bidirectional(GRU(u, activation="tanh", return_sequences=True)),
            Bidirectional(GRU(u, activation="tanh")),
            Dense(CONFIG["prediction_steps"])
        ])
    
        model.compile(
            optimizer=Adam(CONFIG["learning_rate"]),
            loss=Huber(),
            metrics=["mae"]
        )
    
        return model
    
    
    MODEL_BUILDERS = {
        "LSTM": build_LSTM,
        "GRU": build_GRU,
        "BiLSTM": build_BiLSTM,
        "BiGRU": build_BiGRU
    }


    #cell 6
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    
    all_models = {}
    all_histories = {}
    
    for name, builder in MODEL_BUILDERS.items():
    
        print("\n==============================")
        print(f"Training {name}")
        print("==============================")
    
        model = builder()
    
        early_stop = EarlyStopping(
            monitor="val_mae",
            patience=50,
            restore_best_weights=True,
            verbose=1
        )
    
        reduce_lr = ReduceLROnPlateau(
            monitor="val_mae",
            factor=0.7,
            patience=2,
            min_lr=1e-6,
            verbose=1
        )
    
        history = model.fit(
            X_train_all,
            y_train_all,
            validation_data=(X_val_all, y_val_all),
            epochs=80,
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            callbacks=[reduce_lr,early_stop],
            verbose=1
        )
    
        all_models[name] = model
        all_histories[name] = history



    #cell 7

    def compute_metrics(y_true, y_pred):
    
        mae = mean_absolute_error(y_true, y_pred)
    
        mse = mean_squared_error(y_true, y_pred)
    
        rmse = np.sqrt(mse)
    
        y_true = np.clip(y_true,0,None)
        y_pred = np.clip(y_pred,0,None)
    
        rmsle = np.sqrt(
            np.mean(
                (np.log1p(y_pred) - np.log1p(y_true))**2
            )
        )
    
        return {
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "RMSLE": rmsle
        }
    
    
    
    all_results = {}
    all_predictions = {}
    
    for name, model in all_models.items():
    
        print("\n==============================")
        print(f"{name} Centralized Results")
        print("==============================")
    
        all_true_scaled = []
        all_pred_scaled = []
    
        all_true_real = []
        all_pred_real = []
    
        all_predictions[name] = []
    
        for client in clients_data:
    
            y_pred_scaled = model.predict(client["X_test"], verbose=0)
            y_true_scaled = client["y_test"]
    
            scaler = client["scaler"]
    
            y_true_real = scaler.inverse_transform(y_true_scaled)
            y_pred_real = scaler.inverse_transform(y_pred_scaled)
    
            y_true_real = np.clip(y_true_real,0,100)
            y_pred_real = np.clip(y_pred_real,0,100)
    
            all_true_scaled.append(y_true_scaled)
            all_pred_scaled.append(y_pred_scaled)
    
            all_true_real.append(y_true_real)
            all_pred_real.append(y_pred_real)
    
            all_predictions[name].append({
                "true_scaled": y_true_scaled,
                "pred_scaled": y_pred_scaled,
                "true_real": y_true_real,
                "pred_real": y_pred_real
            })
    
    
        y_true_scaled = np.concatenate(all_true_scaled)
        y_pred_scaled = np.concatenate(all_pred_scaled)
    
        y_true_real = np.concatenate(all_true_real)
        y_pred_real = np.concatenate(all_pred_real)
    
        metrics_scaled = compute_metrics(y_true_scaled, y_pred_scaled)
        metrics_real = compute_metrics(y_true_real, y_pred_real)
    
        all_results[name] = {
            "scaled": metrics_scaled,
            "real": metrics_real
        }
    
       #Results
        print(f"MAE_scaled   : {metrics_scaled['MAE']:.6f}")
        print(f"MSE_scaled   : {metrics_scaled['MSE']:.6f}")
        print(f"RMSE_scaled  : {metrics_scaled['RMSE']:.6f}")
        print(f"RMSLE_scaled : {metrics_scaled['RMSLE']:.6f}")
    
        print()
        
        print(f"MAE_real     : {metrics_real['MAE']:.6f}")
        print(f"MSE_real     : {metrics_real['MSE']:.6f}")
        print(f"RMSE_real    : {metrics_real['RMSE']:.6f}")
        print(f"RMSLE_real   : {metrics_real['RMSLE']:.6f}")



    #cell 8
    import json
    
    # CREATE RESULT DIRECTORY
    base_folder = "Materna_results"
    os.makedirs(base_folder, exist_ok=True)
    
    # create dataset name like 291-295results
    dataset_name = f"{min(CLIENT_FILE_NUMBERS)}-{max(CLIENT_FILE_NUMBERS)}_seq{CONFIG['sequence_length']}_results"
    
    run_folder = os.path.join(base_folder, dataset_name)
    os.makedirs(run_folder, exist_ok=True)
    
    
    # SAVE CONFIG USED IN EXPERIMENT
    with open(os.path.join(run_folder, "config.json"), "w") as f:
        json.dump(CONFIG, f, indent=4)
    

    # SAVE RESULTS FOR EACH MODEL
    for model_name in all_results.keys():
    
        print("Saving", model_name)
    
        #save metrics 
        metrics_file = os.path.join(run_folder, f"{model_name}_metrics.json")
    
        with open(metrics_file, "w") as f:
            json.dump(all_results[model_name], f, indent=4)
    
    
        #collect predictions
        rows = []
    
        for client_id, client_pred in enumerate(all_predictions[model_name]):
    
            true_scaled = client_pred["true_scaled"].flatten()
            pred_scaled = client_pred["pred_scaled"].flatten()
    
            true_real = client_pred["true_real"].flatten()
            pred_real = client_pred["pred_real"].flatten()
    
            for i in range(len(true_scaled)):
    
                rows.append({
                    "client": client_id,
                    "true_scaled": true_scaled[i],
                    "pred_scaled": pred_scaled[i],
                    "true_real": true_real[i],
                    "pred_real": pred_real[i]
                })
    
    
        df = pd.DataFrame(rows)
    
        pred_file = os.path.join(run_folder, f"{model_name}_predictions.csv")
    
        df.to_csv(pred_file, index=False)
    
    
    print("\n================================")
    print("Results saved in:", run_folder)
    print("===============================")